## Case-Based Reasoning (CBR) Analisis Putusan Pengadilan

**Sumber data:** [putusan3.mahkamahagung.go.id](https://putusan3.mahkamahagung.go.id)

**Fitur notebook ini:**
- Pilih jenis perkara perdata (Perdata Umum, Perdata Khusus, Perdata Agama)
- Pilih sub-kategori spesifik
- Atur jumlah halaman dan batas unduhan PDF
- Simpan metadata ke CSV
- Upload otomatis ke Google Drive
- Ekstraksi teks dari PDF (untuk CBR)

## Instalasi Library

In [ ]:
!pip install requests beautifulsoup4 lxml pandas tqdm PyPDF2 pdfplumber ipywidgets google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client -q

print("Semua library berhasil diinstall!")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Semua library berhasil diinstall!


## Koneksi Google Drive

In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/CBR_PutusanMA'
    IN_COLAB = True
    print("Google Drive berhasil terhubung!")
    print(f"Data akan disimpan di: {DRIVE_PATH}")
except ImportError:
    # Jika bukan Colab, simpan di folder lokal
    IN_COLAB = False
    DRIVE_PATH = './CBR_PutusanMA'
    print("Tidak terdeteksi sebagai Google Colab.")
    print(f"Data akan disimpan di folder lokal: {DRIVE_PATH}")

# Buat folder struktur
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/pdf", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/metadata", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/teks_ekstrak", exist_ok=True)

print(f"\nStruktur folder:")
print(f"   {DRIVE_PATH}/")
print(f"   ├── pdf/          (file PDF putusan)")
print(f"   ├── metadata/     (CSV daftar putusan)")
print(f"   └── teks_ekstrak/ (teks dari PDF untuk CBR)")

Tidak terdeteksi sebagai Google Colab.
Data akan disimpan di folder lokal: ./CBR_PutusanMA

Struktur folder:
   ./CBR_PutusanMA/
   ├── pdf/          (file PDF putusan)
   ├── metadata/     (CSV daftar putusan)
   └── teks_ekstrak/ (teks dari PDF untuk CBR)


## Import Library & Konfigurasi

In [2]:
import requests
import pandas as pd
import time
import os
import re
import json
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi HTTP headers (meniru browser agar tidak diblok)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8',
    'Referer': 'https://putusan3.mahkamahagung.go.id/'
}

BASE_URL = "https://putusan3.mahkamahagung.go.id"

KATEGORI_PERDATA = {
    "1": {
        "nama": "Perdata Umum",
        "deskripsi": "Gugatan perdata umum: wanprestasi, PMH, waris, tanah, dll.",
        "url_path": "/direktori/index/kategori/perdata-1",
        "sub": {
            "A": {"nama": "Semua Sub-Kategori",       "url_suffix": ""},
            "B": {"nama": "Perbuatan Melawan Hukum",  "url_suffix": "/sub/perbuatan-melawan-hukum"},
            "C": {"nama": "Wanprestasi",              "url_suffix": "/sub/wanprestasi"},
            "D": {"nama": "Sengketa Tanah",           "url_suffix": "/sub/tanah"},
            "E": {"nama": "Waris",                    "url_suffix": "/sub/waris"},
            "F": {"nama": "Perceraian (PN)",          "url_suffix": "/sub/perceraian"},
            "G": {"nama": "Perjanjian",               "url_suffix": "/sub/perjanjian"},
        }
    },
    "2": {
        "nama": "Perdata Khusus",
        "deskripsi": "PHI, Kepailitan, HKI, Niaga, PKPU, dll.",
        "url_path": "/direktori/index/kategori/perdata-khusus",
        "sub": {
            "A": {"nama": "Semua Sub-Kategori",       "url_suffix": ""},
            "B": {"nama": "PHI (Hub. Industrial)",    "url_suffix": "", "url_override": "/direktori/index/kategori/phi"},
            "C": {"nama": "Kepailitan",               "url_suffix": "", "url_override": "/direktori/index/kategori/kepailitan"},
            "D": {"nama": "HKI (Hak Kekayaan Intelektual)", "url_suffix": "", "url_override": "/direktori/index/kategori/hki"},
            "E": {"nama": "PKPU",                     "url_suffix": "", "url_override": "/direktori/index/kategori/pkpu"},
        }
    },
    "3": {
        "nama": "Perdata Agama",
        "deskripsi": "Perceraian, waris Islam, harta bersama, ekonomi syariah.",
        "url_path": "/direktori/index/kategori/perdata-agama-1",
        "sub": {
            "A": {"nama": "Semua Sub-Kategori",       "url_suffix": ""},
            "B": {"nama": "Perceraian",               "url_suffix": "", "url_override": "/direktori/index/kategori/perceraian"},
            "C": {"nama": "Waris Islam",              "url_suffix": "", "url_override": "/direktori/index/kategori/waris-1"},
            "D": {"nama": "Harta Bersama",            "url_suffix": "", "url_override": "/direktori/index/kategori/harta-bersama"},
            "E": {"nama": "Ekonomi Syariah",          "url_suffix": "", "url_override": "/direktori/index/kategori/ekonomi-syariah"},
            "F": {"nama": "Hadhanah (Hak Asuh)",     "url_suffix": "", "url_override": "/direktori/index/kategori/hadhanah"},
        }
    },
}

print("Konfigurasi berhasil dimuat!")
print(f"Base URL: {BASE_URL}")
print(f"Waktu mulai: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Konfigurasi berhasil dimuat!
Base URL: https://putusan3.mahkamahagung.go.id
Waktu mulai: 2026-06-27 19:09:26


## INPUT PENGGUNA: Pilih Jenis Perdata

In [4]:
print("=" * 60)
print("  SCRAPING PUTUSAN MAHKAMAH AGUNG RI")
print("  untuk Case-Based Reasoning (CBR)")
print("=" * 60)
print()
print("PILIH JENIS PERKARA PERDATA:")
print()

for kode, info in KATEGORI_PERDATA.items():
    print(f"  [{kode}] {info['nama']}")
    print(f"       └─ {info['deskripsi']}")
    print()

print("-" * 60)
PILIHAN_KATEGORI = input("Masukkan nomor pilihan (1/2/3): ").strip()

if PILIHAN_KATEGORI not in KATEGORI_PERDATA:
    print("Pilihan tidak valid! Default ke Perdata Umum (1)")
    PILIHAN_KATEGORI = "1"

kategori_terpilih = KATEGORI_PERDATA[PILIHAN_KATEGORI]
print(f"\nKategori dipilih: [{PILIHAN_KATEGORI}] {kategori_terpilih['nama']}")

  SCRAPING PUTUSAN MAHKAMAH AGUNG RI
  untuk Case-Based Reasoning (CBR)

PILIH JENIS PERKARA PERDATA:

  [1] Perdata Umum
       └─ Gugatan perdata umum: wanprestasi, PMH, waris, tanah, dll.

  [2] Perdata Khusus
       └─ PHI, Kepailitan, HKI, Niaga, PKPU, dll.

  [3] Perdata Agama
       └─ Perceraian, waris Islam, harta bersama, ekonomi syariah.

------------------------------------------------------------

Kategori dipilih: [1] Perdata Umum


## INPUT PENGGUNA: Pilih Sub-Kategori & Parameter

In [6]:
print("=" * 60)
print(f"  SUB-KATEGORI: {kategori_terpilih['nama']}")
print("=" * 60)
print()

for kode, sub in kategori_terpilih["sub"].items():
    print(f"  [{kode}] {sub['nama']}")

print()
print("-" * 60)
PILIHAN_SUB = input("Masukkan kode sub-kategori (A/B/C/...): ").strip().upper()

if PILIHAN_SUB not in kategori_terpilih["sub"]:
    print("Pilihan tidak valid! Default ke Semua Sub-Kategori (A)")
    PILIHAN_SUB = "A"

sub_terpilih = kategori_terpilih["sub"][PILIHAN_SUB]
print(f"\nSub-kategori: {sub_terpilih['nama']}")

# Parameter scraping
print()
print("=" * 60)
print("  PARAMETER SCRAPING")
print("=" * 60)
print()
print("Isi parameter di bawah ini (tekan Enter untuk default):")
print()

try:
    JUMLAH_HALAMAN = int(input("  Jumlah halaman yang di-scrape [default=3, max=50]: ").strip() or "3")
    JUMLAH_HALAMAN = min(max(1, JUMLAH_HALAMAN), 50)
except ValueError:
    JUMLAH_HALAMAN = 3

try:
    MAX_PDF = int(input("  Maksimal PDF yang diunduh [default=10, 0=tidak unduh]: ").strip() or "10")
except ValueError:
    MAX_PDF = 10

try:
    DELAY_DETIK = float(input("  Delay antar request dalam detik [default=2]: ").strip() or "2")
    DELAY_DETIK = max(1.0, DELAY_DETIK)  # minimal 1 detik
except ValueError:
    DELAY_DETIK = 2.0

TAHUN_FILTER = input("  Filter tahun putusan [kosongkan = semua, contoh: 2023]: ").strip() or ""
KEYWORD_FILTER = input("  Filter kata kunci dalam judul perkara [kosongkan = semua]: ").strip() or ""
EKSTRAK_TEKS = input("  Ekstrak teks dari PDF untuk CBR? (y/n) [default=y]: ").strip().lower() or "y"
EKSTRAK_TEKS = EKSTRAK_TEKS == "y"

print()
print("=" * 60)
print("  RINGKASAN KONFIGURASI")
print("=" * 60)
print(f"  Kategori      : {kategori_terpilih['nama']}")
print(f"  Sub-Kategori  : {sub_terpilih['nama']}")
print(f"  Halaman       : {JUMLAH_HALAMAN} halaman")
print(f"  Max PDF       : {MAX_PDF} file")
print(f"  Delay         : {DELAY_DETIK} detik")
print(f"  Filter Tahun  : {TAHUN_FILTER or 'Semua tahun'}")
print(f"  Keyword       : {KEYWORD_FILTER or 'Semua'}")
print(f"  Ekstrak Teks  : {'Ya' if EKSTRAK_TEKS else 'Tidak'}")
print(f"  Simpan ke     : {DRIVE_PATH}")
print("=" * 60)

if "url_override" in sub_terpilih and sub_terpilih["url_override"]:
    BASE_SCRAPE_URL = BASE_URL + sub_terpilih["url_override"]
else:
    BASE_SCRAPE_URL = BASE_URL + kategori_terpilih["url_path"] + sub_terpilih.get("url_suffix", "")

if TAHUN_FILTER:
    BASE_SCRAPE_URL = BASE_SCRAPE_URL + f"/tahunjenis/putus/tahun/{TAHUN_FILTER}"

print(f"\nURL Scraping: {BASE_SCRAPE_URL}")
print()
lanjut = input("Lanjutkan scraping? (y/n): ").strip().lower()
if lanjut != 'y':
    print("Scraping dibatalkan.")
else:
    print("Memulai scraping...")

  SUB-KATEGORI: Perdata Umum

  [A] Semua Sub-Kategori
  [B] Perbuatan Melawan Hukum
  [C] Wanprestasi
  [D] Sengketa Tanah
  [E] Waris
  [F] Perceraian (PN)
  [G] Perjanjian

------------------------------------------------------------

Sub-kategori: Semua Sub-Kategori

  PARAMETER SCRAPING

Isi parameter di bawah ini (tekan Enter untuk default):


  RINGKASAN KONFIGURASI
  Kategori      : Perdata Umum
  Sub-Kategori  : Semua Sub-Kategori
  Halaman       : 50 halaman
  Max PDF       : 50 file
  Delay         : 2.0 detik
  Filter Tahun  : Semua tahun
  Keyword       : Semua
  Ekstrak Teks  : Ya
  Simpan ke     : ./CBR_PutusanMA

🌐 URL Scraping: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1

Memulai scraping...


## Fungsi-Fungsi Scraping

In [6]:
def ambil_halaman(url, retries=3):
    """Ambil konten HTML dari URL dengan mekanisme retry."""
    for percobaan in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                return BeautifulSoup(resp.content, 'lxml')
            else:
                print(f"    Status {resp.status_code} pada {url}")
        except requests.exceptions.RequestException as e:
            print(f"    Error (percobaan {percobaan+1}/{retries}): {e}")
            time.sleep(DELAY_DETIK * 2)
    return None


def parse_daftar_putusan(soup):
    """Ekstrak metadata putusan dari halaman direktori."""
    putusan_list = []
    
    items = soup.find_all('div', class_='spost') or soup.find_all('article')
    
    if not items:
        links = soup.find_all('a', href=re.compile(r'/direktori/putusan/'))
        for link in links:
            href = link.get('href', '')
            teks = link.get_text(strip=True)
            if 'Putusan' in teks or 'Nomor' in teks:
                putusan_list.append({
                    'judul': teks,
                    'url_detail': BASE_URL + href if href.startswith('/') else href,
                    'nomor': ekstrak_nomor(teks),
                    'tanggal': '',
                    'pengadilan': '',
                    'para_pihak': '',
                    'klasifikasi': sub_terpilih['nama'],
                })
        return putusan_list
    
    for item in items:
        try:
            judul_el = item.find('h2') or item.find('h3') or item.find('strong')
            link_el  = item.find('a', href=re.compile(r'/direktori/putusan/'))
            
            teks_item = item.get_text(separator=' | ', strip=True)
            
            putusan = {
                'judul'       : judul_el.get_text(strip=True) if judul_el else teks_item[:100],
                'url_detail'  : (BASE_URL + link_el['href'] if link_el else ''),
                'nomor'       : ekstrak_nomor(teks_item),
                'tanggal'     : ekstrak_tanggal(teks_item),
                'pengadilan'  : ekstrak_pengadilan(teks_item),
                'para_pihak'  : ekstrak_para_pihak(teks_item),
                'klasifikasi' : sub_terpilih['nama'],
                'kategori'    : kategori_terpilih['nama'],
            }
            
            if KEYWORD_FILTER:
                gabung = (putusan['judul'] + ' ' + putusan['para_pihak']).lower()
                if KEYWORD_FILTER.lower() not in gabung:
                    continue
            
            putusan_list.append(putusan)
        except Exception as e:
            continue
    
    return putusan_list


def parse_halaman_direktori_baru(soup):
    """Parser untuk format terbaru halaman direktori MA."""
    putusan_list = []
    
    teks_halaman = soup.get_text()
    semua_link = soup.find_all('a', href=True)
    
    for link in semua_link:
        href = link['href']
        if '/direktori/putusan/' not in href:
            continue
        
        teks_link = link.get_text(strip=True)
        if not teks_link:
            continue
        
        parent = link.parent
        teks_konteks = parent.get_text(separator=' ', strip=True) if parent else teks_link
        
        try:
            gp = parent.parent
            teks_konteks = gp.get_text(separator=' | ', strip=True)[:500]
        except:
            pass
        
        full_url = BASE_URL + href if href.startswith('/') else href
        
        putusan = {
            'judul'       : teks_link,
            'url_detail'  : full_url,
            'nomor'       : ekstrak_nomor(teks_konteks),
            'tanggal'     : ekstrak_tanggal(teks_konteks),
            'pengadilan'  : ekstrak_pengadilan(teks_konteks),
            'para_pihak'  : ekstrak_para_pihak(teks_konteks),
            'klasifikasi' : sub_terpilih['nama'],
            'kategori'    : kategori_terpilih['nama'],
            'teks_konteks': teks_konteks
        }
        
        if KEYWORD_FILTER:
            gabung = (putusan['judul'] + ' ' + putusan['para_pihak']).lower()
            if KEYWORD_FILTER.lower() not in gabung:
                continue
        
        putusan_list.append(putusan)
    
    seen = set()
    unik = []
    for p in putusan_list:
        if p['url_detail'] not in seen and p['url_detail']:
            seen.add(p['url_detail'])
            unik.append(p)
    
    return unik


def ekstrak_nomor(teks):
    """Ekstrak nomor putusan dari teks."""
    pola = re.search(r'Nomor\s+([\w\s/.-]+?)(?:\s*·|\s*Tanggal|\s*—|$)', teks, re.IGNORECASE)
    if pola:
        return pola.group(1).strip()[:80]
    pola2 = re.search(r'(\d+\s*/\s*(?:PDT|Pdt|AG|PID)[\w./\s-]*)', teks)
    return pola2.group(1).strip() if pola2 else ''


def ekstrak_tanggal(teks):
    """Ekstrak tanggal putus dari teks."""
    pola = re.search(r'(?:Putus|Tanggal)\s*[:\-–]?\s*(\d{1,2}\s+\w+\s+\d{4})', teks, re.IGNORECASE)
    if pola:
        return pola.group(1).strip()
    pola2 = re.search(r'(\d{1,2}-\d{2}-\d{4})', teks)
    return pola2.group(1) if pola2 else ''


def ekstrak_pengadilan(teks):
    """Ekstrak nama pengadilan dari teks."""
    pola = re.search(r'Pengadilan\s+([A-Z\s]+?)(?:\s+Perdata|\s+Pidana|\s*·|\s*$)', teks)
    if pola:
        return pola.group(1).strip()[:60]
    if 'MAHKAMAH AGUNG' in teks.upper():
        return 'MAHKAMAH AGUNG'
    return ''


def ekstrak_para_pihak(teks):
    """Ekstrak nama para pihak (penggugat vs tergugat)."""
    pola = re.search(r'([A-Z][\w\s,.]+?)\s+(?:VS|vs|melawan|MELAWAN)\s+([A-Z][\w\s,.]+?)(?:\s*·|\s*\d+\s*—|$)', teks)
    if pola:
        penggugat = pola.group(1).strip()[:100]
        tergugat  = pola.group(2).strip()[:100]
        return f"{penggugat} VS {tergugat}"
    return ''


def ambil_url_pdf(url_detail):
    """Ambil URL download PDF dari halaman detail putusan."""
    soup = ambil_halaman(url_detail)
    if not soup:
        return None
    
    # Cari link PDF
    for link in soup.find_all('a', href=True):
        href = link['href']
        teks = link.get_text(strip=True).lower()
        if href.endswith('.pdf') or 'pdf' in href.lower() or 'download' in teks:
            return BASE_URL + href if href.startswith('/') else href
    
    # Cari di meta atau data attributes
    for tag in soup.find_all(attrs={"data-pdf": True}):
        return tag['data-pdf']
    
    return None


def unduh_pdf(url_pdf, nama_file, folder):
    """Unduh file PDF dan simpan ke folder."""
    path_file = os.path.join(folder, nama_file)
    
    if os.path.exists(path_file):
        return path_file  # Sudah ada, skip
    
    try:
        resp = requests.get(url_pdf, headers=HEADERS, timeout=60, stream=True)
        if resp.status_code == 200:
            with open(path_file, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return path_file
    except Exception as e:
        print(f"    Gagal unduh PDF: {e}")
    return None


def ekstrak_teks_pdf(path_pdf):
    """Ekstrak teks dari file PDF untuk digunakan dalam CBR."""
    teks = ""
    try:
        import pdfplumber
        with pdfplumber.open(path_pdf) as pdf:
            for halaman in pdf.pages[:10]:  # Maksimal 10 halaman pertama
                t = halaman.extract_text()
                if t:
                    teks += t + "\n"
    except Exception:
        try:
            import PyPDF2
            with open(path_pdf, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                for halaman in reader.pages[:10]:
                    teks += halaman.extract_text() or ""
        except Exception as e:
            teks = f"[Gagal ekstrak: {e}]"
    return teks.strip()


def buat_nama_file_aman(teks, max_len=60):
    """Buat nama file yang aman dari teks nomor putusan."""
    aman = re.sub(r'[<>:"/\\|?*]', '_', teks)
    aman = re.sub(r'\s+', '_', aman)
    return aman[:max_len]


print("✅ Semua fungsi scraping berhasil dimuat!")

✅ Semua fungsi scraping berhasil dimuat!


## Mulai Scraping Metadata

In [7]:
print("=" * 60)
print("  FASE 1: SCRAPING METADATA PUTUSAN")
print("=" * 60)
print()

semua_putusan = []
gagal_halaman = []

for nomor_halaman in tqdm(range(1, JUMLAH_HALAMAN + 1), desc="📄 Scraping halaman"):
    
    # Bangun URL halaman
    if nomor_halaman == 1:
        url_halaman = f"{BASE_SCRAPE_URL}.html"
    else:
        url_halaman = f"{BASE_SCRAPE_URL}/page/{nomor_halaman}.html"
    
    print(f"\n  Halaman {nomor_halaman}: {url_halaman}")
    
    soup = ambil_halaman(url_halaman)
    
    if not soup:
        print(f"  Gagal mengambil halaman {nomor_halaman}")
        gagal_halaman.append(nomor_halaman)
        continue
    
    # Parse putusan dari halaman
    putusan_halaman = parse_halaman_direktori_baru(soup)
    
    print(f"  Ditemukan {len(putusan_halaman)} putusan di halaman ini")
    
    if not putusan_halaman:
        print(f"    Tidak ada data di halaman {nomor_halaman}, mungkin sudah halaman terakhir.")
        break
    
    semua_putusan.extend(putusan_halaman)
    
    # Delay antar halaman
    time.sleep(DELAY_DETIK)

print()
print("=" * 60)
print(f"    TOTAL: {len(semua_putusan)} putusan berhasil dikumpulkan")
if gagal_halaman:
    print(f"    Halaman gagal diambil: {gagal_halaman}")
print("=" * 60)

# Preview 5 data pertama
if semua_putusan:
    print("\n   Preview 5 data pertama:")
    for i, p in enumerate(semua_putusan[:5], 1):
        print(f"  {i}. {p.get('judul', 'N/A')[:80]}")
        print(f"     Nomor  : {p.get('nomor', 'N/A')}")
        print(f"     Tanggal: {p.get('tanggal', 'N/A')}")
        print(f"     URL    : {p.get('url_detail', 'N/A')[:70]}")
        print()

  FASE 1: SCRAPING METADATA PUTUSAN



📄 Scraping halaman:   0%|          | 0/3 [00:00<?, ?it/s]


  Halaman 1: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1.html
    Status 403 pada https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1.html
    Status 403 pada https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1.html
    Status 403 pada https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1.html
  Gagal mengambil halaman 1

  Halaman 2: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1/page/2.html
    Status 403 pada https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1/page/2.html
    Status 403 pada https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1/page/2.html
    Status 403 pada https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1/page/2.html
  Gagal mengambil halaman 2

  Halaman 3: https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1/page/3.html
    Status 403 pada https://putusan3.mahkamahagung.go.

## Simpan Metadata ke CSV

In [ ]:
print("=" * 60)
print("  FASE 2: SIMPAN METADATA KE CSV")
print("=" * 60)

if not semua_putusan:
    print("Tidak ada data untuk disimpan.")
else:
    df = pd.DataFrame(semua_putusan)
    
    # Tambah kolom tambahan
    df['timestamp_scrape'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    df['id_unik'] = range(1, len(df) + 1)
    
    # Reorder kolom
    kolom_urut = ['id_unik', 'nomor', 'judul', 'tanggal', 'pengadilan', 
                  'para_pihak', 'kategori', 'klasifikasi', 'url_detail', 
                  'timestamp_scrape']
    kolom_ada = [k for k in kolom_urut if k in df.columns]
    df = df[kolom_ada]
    
    # Nama file CSV
    nama_kategori_file = buat_nama_file_aman(f"{kategori_terpilih['nama']}_{sub_terpilih['nama']}")
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    nama_csv = f"metadata_{nama_kategori_file}_{timestamp}.csv"
    path_csv = os.path.join(DRIVE_PATH, 'metadata', nama_csv)
    
    df.to_csv(path_csv, index=False, encoding='utf-8-sig')
    
    print(f"\nMetadata berhasil disimpan!")
    print(f"    File  : {nama_csv}")
    print(f"    Path  : {path_csv}")
    print(f"    Total : {len(df)} baris data")
    print()
    
    # Tampilkan statistik
    print("Statistik data:")
    print(df.describe(include='all').T[['count', 'unique', 'top']].to_string())
    print()
    print("\nPreview DataFrame:")
    display(df.head(10))

## Unduh PDF Putusan

In [ ]:
print("=" * 60)
print("  FASE 3: UNDUH FILE PDF PUTUSAN")
print("=" * 60)

if MAX_PDF == 0:
    print("Pengunduhan PDF dilewati (MAX_PDF = 0)")
elif not semua_putusan:
    print("Tidak ada data untuk diunduh.")
else:
    folder_pdf   = os.path.join(DRIVE_PATH, 'pdf')
    folder_teks  = os.path.join(DRIVE_PATH, 'teks_ekstrak')
    
    log_unduhan = []
    jumlah_berhasil = 0
    jumlah_gagal    = 0
    
    target = semua_putusan[:MAX_PDF]
    print(f"\nTarget: {len(target)} PDF dari {len(semua_putusan)} putusan\n")
    
    for i, putusan in enumerate(tqdm(target, desc="Unduh PDF"), 1):
        url_detail = putusan.get('url_detail', '')
        nomor      = putusan.get('nomor', f'putusan_{i}')
        
        if not url_detail:
            jumlah_gagal += 1
            continue
        
        print(f"\n  [{i}/{len(target)}] {nomor[:60]}")
        print(f"    {url_detail[:70]}")
        
        # Ambil URL PDF dari halaman detail
        url_pdf = ambil_url_pdf(url_detail)
        time.sleep(DELAY_DETIK)
        
        if not url_pdf:
            print(f"URL PDF tidak ditemukan")
            jumlah_gagal += 1
            log_unduhan.append({**putusan, 'status_pdf': 'tidak_ada_link', 'path_pdf': ''})
            continue
        
        # Unduh PDF
        nama_file_pdf = buat_nama_file_aman(nomor or f"putusan_{i}") + ".pdf"
        path_pdf = unduh_pdf(url_pdf, nama_file_pdf, folder_pdf)
        time.sleep(DELAY_DETIK)
        
        if path_pdf:
            print(f"PDF tersimpan: {nama_file_pdf}")
            jumlah_berhasil += 1
            
            # Ekstrak teks untuk CBR
            path_teks = ''
            if EKSTRAK_TEKS:
                teks = ekstrak_teks_pdf(path_pdf)
                if teks:
                    nama_file_teks = buat_nama_file_aman(nomor or f"putusan_{i}") + ".txt"
                    path_teks = os.path.join(folder_teks, nama_file_teks)
                    with open(path_teks, 'w', encoding='utf-8') as f:
                        f.write(f"NOMOR PUTUSAN: {nomor}\n")
                        f.write(f"TANGGAL: {putusan.get('tanggal', '')}\n")
                        f.write(f"PENGADILAN: {putusan.get('pengadilan', '')}\n")
                        f.write(f"PARA PIHAK: {putusan.get('para_pihak', '')}\n")
                        f.write(f"KATEGORI: {putusan.get('kategori', '')}\n")
                        f.write(f"URL: {url_detail}\n")
                        f.write("=" * 60 + "\n")
                        f.write(teks)
                    print(f"    Teks diekstrak: {nama_file_teks}")
            
            log_unduhan.append({
                **putusan,
                'url_pdf'    : url_pdf,
                'path_pdf'   : path_pdf,
                'path_teks'  : path_teks,
                'status_pdf' : 'berhasil'
            })
        else:
            print(f"Gagal mengunduh PDF")
            jumlah_gagal += 1
            log_unduhan.append({**putusan, 'status_pdf': 'gagal_unduh', 'path_pdf': ''})
    
    # Simpan log unduhan
    if log_unduhan:
        df_log = pd.DataFrame(log_unduhan)
        nama_log = f"log_unduhan_{timestamp}.csv"
        path_log = os.path.join(DRIVE_PATH, 'metadata', nama_log)
        df_log.to_csv(path_log, index=False, encoding='utf-8-sig')
        print(f"\nLog unduhan disimpan: {nama_log}")
    
    print()
    print("=" * 60)
    print(f"    RINGKASAN UNDUHAN PDF")
    print(f"    Berhasil : {jumlah_berhasil} file")
    print(f"    Gagal    : {jumlah_gagal} file")
    print(f"    Folder  : {folder_pdf}")
    if EKSTRAK_TEKS:
        print(f"    Teks CBR: {folder_teks}")
    print("=" * 60)

## Pra-Pemrosesan untuk CBR (Case-Based Reasoning)

In [ ]:
print("=" * 60)
print("  FASE 4: PRA-PEMROSESAN UNTUK CBR")
print("=" * 60)
print()
print("Membangun struktur kasus (case base) untuk CBR...")
print()

import json

def bangun_case_base(folder_teks, metadata_df=None):
    """
    Bangun case base dari file teks yang sudah diekstrak.
    Setiap kasus memiliki:
    - problem: deskripsi masalah hukum
    - solution: amar putusan
    - outcome: hasil (menang/kalah/sebagian)
    - features: fitur-fitur kasus untuk CBR
    """
    case_base = []
    
    file_teks_list = [f for f in os.listdir(folder_teks) if f.endswith('.txt')]
    
    if not file_teks_list:
        print("Belum ada file teks. Jalankan Cell 9 terlebih dahulu.")
        return case_base
    
    print(f"Memproses {len(file_teks_list)} file teks...")
    
    for nama_file in tqdm(file_teks_list, desc="🔍 Proses CBR"):
        path_file = os.path.join(folder_teks, nama_file)
        
        try:
            with open(path_file, 'r', encoding='utf-8') as f:
                isi = f.read()
            
            # Ekstrak header metadata
            baris = isi.split('\n')
            meta = {}
            teks_isi = []
            setelah_garis = False
            
            for baris_i in baris:
                if '=' * 10 in baris_i:
                    setelah_garis = True
                    continue
                if not setelah_garis and ':' in baris_i:
                    kunci, _, nilai = baris_i.partition(':')
                    meta[kunci.strip()] = nilai.strip()
                elif setelah_garis:
                    teks_isi.append(baris_i)
            
            teks_lengkap = '\n'.join(teks_isi)
            
            # Ekstrak komponen untuk CBR
            amar = ekstrak_amar(teks_lengkap)
            pokok_perkara = ekstrak_pokok_perkara(teks_lengkap)
            dasar_hukum = ekstrak_dasar_hukum(teks_lengkap)
            
            kasus = {
                'case_id'      : nama_file.replace('.txt', ''),
                'nomor'        : meta.get('NOMOR PUTUSAN', ''),
                'tanggal'      : meta.get('TANGGAL', ''),
                'pengadilan'   : meta.get('PENGADILAN', ''),
                'para_pihak'   : meta.get('PARA PIHAK', ''),
                'kategori'     : meta.get('KATEGORI', ''),
                'url'          : meta.get('URL', ''),
                # Komponen CBR
                'problem'      : pokok_perkara[:500],
                'solution'     : amar[:500],
                'dasar_hukum'  : dasar_hukum[:300],
                'teks_lengkap' : teks_lengkap[:2000],
                # Fitur untuk similarity matching
                'fitur': {
                    'jenis_perkara' : meta.get('KATEGORI', ''),
                    'pengadilan'    : meta.get('PENGADILAN', ''),
                    'ada_amar'      : bool(amar),
                    'panjang_teks'  : len(teks_lengkap),
                    'kata_kunci'    : ekstrak_kata_kunci(teks_lengkap),
                }
            }
            
            case_base.append(kasus)
            
        except Exception as e:
            print(f"Gagal proses {nama_file}: {e}")
    
    return case_base


def ekstrak_amar(teks):
    """Ekstrak amar putusan dari teks."""
    pola_amar = re.search(
        r'(?:MENGADILI|M E N G A D I L I|Amar Putusan)(.*?)(?:Demikian|PENUTUP|$)',
        teks, re.DOTALL | re.IGNORECASE
    )
    if pola_amar:
        return pola_amar.group(1).strip()[:500]
    
    # Cari pola mengabulkan/menolak
    for pola in [r'(Mengabulkan gugatan.*?;)', r'(Menolak gugatan.*?;)', 
                 r'(Menyatakan.*?terbukti.*?;)']:
        m = re.search(pola, teks, re.IGNORECASE | re.DOTALL)
        if m:
            return m.group(1).strip()[:300]
    return ''


def ekstrak_pokok_perkara(teks):
    """Ekstrak pokok perkara dari teks."""
    for pola in [
        r'(?:DUDUK PERKARA|Duduk Perkara|TENTANG DUDUK)(.*?)(?:TENTANG HUKUM|PERTIMBANGAN|$)',
        r'(?:bahwa|Bahwa)\s+(penggugat|pemohon|tergugat).*?(?:;|\n\n)'
    ]:
        m = re.search(pola, teks, re.DOTALL | re.IGNORECASE)
        if m:
            return m.group(1).strip()[:500]
    return teks[:300]  # Fallback: 300 karakter pertama


def ekstrak_dasar_hukum(teks):
    """Ekstrak dasar hukum / pasal yang dikutip."""
    pasal_list = re.findall(
        r'(?:Pasal|pasal)\s+\d+[\s\w]*(?:KUHPerdata|KUH Perdata|HIR|RBg|UU|Undang-Undang)[\w\s.-]*',
        teks
    )
    return ' | '.join(list(set(pasal_list))[:10])


def ekstrak_kata_kunci(teks, top_n=10):
    """Ekstrak kata kunci penting dari teks."""
    # Kata-kata stopword sederhana bahasa Indonesia
    stopwords = {'yang', 'dan', 'di', 'ke', 'dari', 'untuk', 'dengan', 'pada', 
                 'adalah', 'ini', 'itu', 'dalam', 'oleh', 'sebagai', 'atau',
                 'bahwa', 'telah', 'tidak', 'akan', 'juga', 'serta', 'tersebut',
                 'dapat', 'harus', 'sudah', 'atas', 'maka', 'kepada', 'hal'}
    
    kata = re.findall(r'\b[a-zA-Z]{4,}\b', teks.lower())
    kata_bersih = [k for k in kata if k not in stopwords]
    
    from collections import Counter
    counter = Counter(kata_bersih)
    return [k for k, _ in counter.most_common(top_n)]


# Jalankan pembuatan case base
folder_teks = os.path.join(DRIVE_PATH, 'teks_ekstrak')
case_base = bangun_case_base(folder_teks)

if case_base:
    # Simpan case base ke JSON
    path_case_base = os.path.join(DRIVE_PATH, 'metadata', f'case_base_{timestamp}.json')
    with open(path_case_base, 'w', encoding='utf-8') as f:
        json.dump(case_base, f, ensure_ascii=False, indent=2)
    
    # Simpan juga ke CSV
    df_cbr = pd.DataFrame([{
        'case_id'     : k['case_id'],
        'nomor'       : k['nomor'],
        'tanggal'     : k['tanggal'],
        'pengadilan'  : k['pengadilan'],
        'para_pihak'  : k['para_pihak'],
        'problem'     : k['problem'],
        'solution'    : k['solution'],
        'dasar_hukum' : k['dasar_hukum'],
        'kata_kunci'  : str(k['fitur']['kata_kunci']),
    } for k in case_base])
    
    path_cbr_csv = os.path.join(DRIVE_PATH, 'metadata', f'case_base_{timestamp}.csv')
    df_cbr.to_csv(path_cbr_csv, index=False, encoding='utf-8-sig')
    
    print(f"\nCase base berhasil dibangun!")
    print(f"Total kasus : {len(case_base)}")
    print(f"JSON        : {path_case_base}")
    print(f"CSV         : {path_cbr_csv}")
    print()
    print(f"Preview case base:")
    display(df_cbr.head())
else:
    print("Case base kosong. Unduh PDF terlebih dahulu di Cell 9.")

## Demo CBR: Cari Kasus Serupa

In [ ]:
print("=" * 60)
print("  DEMO CBR: CARI KASUS SERUPA")
print("=" * 60)
print()
print("Modul ini mendemonstrasikan pencarian kasus serupa")
print("menggunakan pendekatan CBR berbasis keyword similarity.")
print()

def hitung_similaritas(kasus_query, kasus_db, bobot=None):
    """
    Hitung similaritas antara kasus query dan kasus di database.
    Menggunakan Weighted Feature Matching sederhana.
    
    Komponen similarity:
    1. Kecocokan kata kunci (40%)
    2. Kecocokan jenis perkara (30%)
    3. Kecocokan pengadilan (15%)
    4. Kesamaan teks (15%)
    """
    if bobot is None:
        bobot = {
            'kata_kunci'  : 0.40,
            'jenis_perkara': 0.30,
            'pengadilan'  : 0.15,
            'teks'        : 0.15,
        }
    
    skor = 0.0
    
    # 1. Kecocokan kata kunci
    kk_query = set(kasus_query.get('kata_kunci', []))
    kk_db    = set(kasus_db.get('fitur', {}).get('kata_kunci', []))
    if kk_query and kk_db:
        irisan = kk_query & kk_db
        gabung = kk_query | kk_db
        skor_kk = len(irisan) / len(gabung) if gabung else 0
        skor += bobot['kata_kunci'] * skor_kk
    
    # 2. Kecocokan jenis perkara
    if kasus_query.get('jenis_perkara', '').lower() == kasus_db.get('kategori', '').lower():
        skor += bobot['jenis_perkara']
    
    # 3. Kecocokan pengadilan
    if kasus_query.get('pengadilan', '').lower() in kasus_db.get('pengadilan', '').lower():
        skor += bobot['pengadilan']
    
    # 4. Kesamaan teks sederhana (berapa % kata yang sama)
    kata_query = set(re.findall(r'\b[a-zA-Z]{4,}\b', kasus_query.get('deskripsi', '').lower()))
    kata_db    = set(re.findall(r'\b[a-zA-Z]{4,}\b', kasus_db.get('teks_lengkap', '')[:500].lower()))
    if kata_query and kata_db:
        irisan_t = kata_query & kata_db
        gabung_t = kata_query | kata_db
        skor += bobot['teks'] * (len(irisan_t) / len(gabung_t) if gabung_t else 0)
    
    return round(skor, 4)


def cari_kasus_serupa(deskripsi_kasus, jenis_perkara='', pengadilan='', top_k=5):
    """
    Cari kasus-kasus serupa dari case base.
    
    Args:
        deskripsi_kasus: Deskripsi kasus yang ingin dicari padanannya
        jenis_perkara  : Jenis perkara (opsional)
        pengadilan     : Nama pengadilan (opsional)
        top_k          : Jumlah hasil teratas yang dikembalikan
    
    Returns:
        List kasus paling mirip beserta skor similaritas
    """
    if not case_base:
        print("Case base kosong. Silakan unduh PDF dan bangun case base terlebih dahulu.")
        return []
    
    # Siapkan query
    kata_query = ekstrak_kata_kunci(deskripsi_kasus, top_n=15)
    query = {
        'deskripsi'    : deskripsi_kasus,
        'kata_kunci'   : kata_query,
        'jenis_perkara': jenis_perkara,
        'pengadilan'   : pengadilan,
    }
    
    # Hitung skor untuk setiap kasus
    hasil = []
    for kasus in case_base:
        skor = hitung_similaritas(query, kasus)
        hasil.append({
            'skor'       : skor,
            'nomor'      : kasus.get('nomor', ''),
            'tanggal'    : kasus.get('tanggal', ''),
            'pengadilan' : kasus.get('pengadilan', ''),
            'para_pihak' : kasus.get('para_pihak', ''),
            'problem'    : kasus.get('problem', '')[:200],
            'solution'   : kasus.get('solution', '')[:200],
            'url'        : kasus.get('url', ''),
        })
    
    # Urutkan berdasarkan skor tertinggi
    hasil.sort(key=lambda x: x['skor'], reverse=True)
    return hasil[:top_k]


# ── Demo interaktif ─────────────────────────────────────────────
print("Masukkan deskripsi kasus yang ingin Anda cari padanannya:")
print("(Contoh: 'sengketa tanah waris antara ahli waris atas sertifikat hak milik')")
print()

DESKRIPSI_QUERY  = input("  Deskripsi kasus: ").strip()
FILTER_JENIS     = input("  Filter jenis perkara (kosongkan = semua): ").strip()
FILTER_PENGADILAN= input("  Filter pengadilan (kosongkan = semua): ").strip()
TOP_K            = int(input("  Tampilkan berapa hasil teratas [default=5]: ").strip() or "5")

if DESKRIPSI_QUERY:
    print()
    print("=" * 60)
    print("  HASIL PENCARIAN CBR")
    print("=" * 60)
    print(f"  Query: {DESKRIPSI_QUERY[:80]}")
    print()
    
    hasil_cbr = cari_kasus_serupa(
        DESKRIPSI_QUERY, 
        FILTER_JENIS, 
        FILTER_PENGADILAN, 
        TOP_K
    )
    
    if hasil_cbr:
        for i, kasus in enumerate(hasil_cbr, 1):
            print(f"  [{i}] Skor Similaritas: {kasus['skor']:.2%}")
            print(f"      Nomor     : {kasus['nomor']}")
            print(f"      Tanggal   : {kasus['tanggal']}")
            print(f"      Pengadilan: {kasus['pengadilan']}")
            print(f"      Para Pihak: {kasus['para_pihak'][:60]}")
            print(f"      Masalah   : {kasus['problem'][:100]}...")
            print(f"      Solusi    : {kasus['solution'][:100]}...")
            print(f"      URL       : {kasus['url'][:70]}")
            print()
        
        # Simpan hasil CBR
        df_hasil = pd.DataFrame(hasil_cbr)
        path_hasil = os.path.join(DRIVE_PATH, 'metadata', f'hasil_cbr_{timestamp}.csv')
        df_hasil.to_csv(path_hasil, index=False, encoding='utf-8-sig')
        print(f"Hasil disimpan ke: {path_hasil}")
    else:
        print("Tidak ditemukan kasus serupa.")
else:
    print("Query kosong, demo CBR dilewati.")

## Ringkasan Akhir & Daftar File

In [ ]:
print("=" * 60)
print("  RINGKASAN AKHIR")
print("=" * 60)
print()
print(f"Kategori      : {kategori_terpilih['nama']} — {sub_terpilih['nama']}")
print(f"Metadata      : {len(semua_putusan)} putusan")
print(f"Case base CBR : {len(case_base)} kasus")
print(f"Lokasi data   : {DRIVE_PATH}")
print()

# Daftar semua file yang dibuat
print("Daftar file yang dibuat:")
for root, dirs, files in os.walk(DRIVE_PATH):
    level = root.replace(DRIVE_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for file in files:
        ukuran = os.path.getsize(os.path.join(root, file))
        if ukuran > 1024*1024:
            ukuran_str = f"{ukuran/1024/1024:.1f} MB"
        elif ukuran > 1024:
            ukuran_str = f"{ukuran/1024:.1f} KB"
        else:
            ukuran_str = f"{ukuran} B"
        print(f"{sub_indent}{file} ({ukuran_str})")

print()
print("=" * 60)
print("  🎓 Siap digunakan untuk analisis CBR!")
print("  Gunakan file case_base_*.json/csv sebagai input CBR.")
print("=" * 60)